# Sample 1 — every mistake, model by model

The annotation for sample 1 lists a fixed set of items. For each model below:
what it got right, and every single thing it got wrong.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import (
    _confusion_from_match, _match_structured, extract_gold, micro_prf1,
    resolve_old_gt_path,
)

SAMPLE, EXTRACTOR = 1, 'pdfplumber'
pd.set_option('display.max_colwidth', 90)
INK, WRONG = '#111111', '#fdecea'      # wrong rows tinted, right rows left plain
gold = extract_gold(resolve_old_gt_path(SAMPLE))


def mistakes(model):
    """Every wrong item this model produced, and the score for the document."""
    tag = P.make_tag(model, EXTRACTOR)
    rec, extra = _match_structured(P.structured_path(tag, SAMPLE), gold)
    m = micro_prf1(_confusion_from_match(rec, extra))
    rows = [{'should be': r['gold_label'], 'model said': r['pred_label'],
             'text': r['pred_text']}
            for r in rec if r['pred_label'] and r['pred_label'] != r['gold_label']]
    rows += [{'should be': 'nothing — extra item', 'model said': l, 'text': t}
             for t, l in extra]
    rows += [{'should be': r['gold_label'], 'model said': 'not produced',
              'text': r['gold_text']}
             for r in rec if r['pred_label'] is None]
    return pd.DataFrame(rows), m


def side_by_side(model):
    """Every annotation item next to the label the model gave that same text.

    Matching is by shared words, not by row number — the model often produces
    a different number of items, so the two lists drift out of step and
    comparing row 12 with row 12 would compare unrelated things.
    """
    tag = P.make_tag(model, EXTRACTOR)
    rec, _ = _match_structured(P.structured_path(tag, SAMPLE), gold)
    return pd.DataFrame([{
        'annotation says': r['gold_label'],
        'model said': r['pred_label'] if r['pred_label'] else 'not produced',
        'ok': '' if r['pred_label'] == r['gold_label'] else 'X',
        'text': r['gold_text'],
    } for r in rec])


def flag(row):
    """Tint the whole row when the model got it wrong."""
    bad = row['ok'] == 'X'
    return [f'background-color: {WRONG}; color: {INK}' if bad else '' for _ in row]


def show(model):
    """Score, then every item, then the extra items."""
    df, m = mistakes(model)
    print(f'{len(gold)} items in the annotation')
    print(f'  {m["tp"]:>2} correct')
    print(f'  {len(df):>2} wrong')
    print(f'  f1 = {m["f1"]:.3f}')

    print('\nEvery item, annotation against model:')
    display(side_by_side(model).style.apply(flag, axis=1))

    tag = P.make_tag(model, EXTRACTOR)
    _, extra = _match_structured(P.structured_path(tag, SAMPLE), gold)
    if extra:
        print(f'Plus {len(extra)} item(s) the model produced that the annotation '
              f'does not contain:')
        display(pd.DataFrame([{'model said': l, 'text': t} for t, l in extra]))


print('sections below:', ', '.join(['llama3.1:8b', 'gemma4:e4b', 'llama3.3:70b']))


sections below: llama3.1:8b, gemma4:e4b, llama3.3:70b


## llama3.1:8b


In [2]:
show('llama3.1:8b')


28 items in the annotation
  19 correct
  12 wrong
  f1 = 0.644

Every item, annotation against model:


,annotation says,model said,ok,text
0,title,title,,DATA MANAGEMENT AND SHARING PLAN
1,section.title,section.title,,Element 1: Data Type:
2,question.text,question.text,,A. Types and amount of scientific data expected to be generated in the project:
3,answer.text,answer.text,,"This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and the publicly available NHANES cohorts (wrist NHANES 2011-2014; hip/counts-NHANES 2003-2006). The studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the iWATCH Study, (iv) the MOCA Study, (v) the PHASE Study, (vi) the AusDiab Study, (vii) the ACT Study, and (viii) the WHISH accelerometer substudy."
4,question.text,section.title,X,"B. Scientific data that will be preserved and shared, and the rationale for doing so:"
5,answer.text,answer.text,,"As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Repository sitting behavior metrics or analysis that were created in this project from the following UCSD-based study datasets: (i) The RISE study (ii) The iWATCH study (iii) NHANES cohorts Sufficient data from those datasets will be preserved to enable sharing to validate and replicate research findings described in the Aims with those datasets only. Please see Element 5 for data access information about other studies."
6,question.text,section.title,X,"C. Metadata, other relevant data, and associated documentation:"
7,answer.text,answer.text,,"In addition to the data described above, code and models will be included in the repository and in on our project's GitHub website, hosted as part of the UCSD Advance Data Analytics Lab GitHub and titled ""Deep Postures""."
8,section.title,section.title,,"Element 2: Related Tools, Software and/or Code:"
9,answer.text,answer.text,,"Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using ActiLife software which requires a paid license. All custom code and models will also be shared in the data repository and on our project's GitHub website, hosted as part of the UCSD Advance Data Analytics Lab GitHub and titled ""Deep Postures""."


Plus 3 item(s) the model produced that the annotation does not contain:


,model said,text
0,answer.text,Objective sedentary behavior metrics – no existing standards
1,question.text,about using their data:
2,answer.text,(i) the SOL-VIDA Study – Data requests may be made by following instructions on the Hi...


## gemma4:e4b


In [3]:
show('gemma4:e4b')


28 items in the annotation
  28 correct
   0 wrong
  f1 = 1.000

Every item, annotation against model:


,annotation says,model said,ok,text
0,title,title,,DATA MANAGEMENT AND SHARING PLAN
1,section.title,section.title,,Element 1: Data Type:
2,question.text,question.text,,A. Types and amount of scientific data expected to be generated in the project:
3,answer.text,answer.text,,"This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and the publicly available NHANES cohorts (wrist NHANES 2011-2014; hip/counts-NHANES 2003-2006). The studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the iWATCH Study, (iv) the MOCA Study, (v) the PHASE Study, (vi) the AusDiab Study, (vii) the ACT Study, and (viii) the WHISH accelerometer substudy."
4,question.text,question.text,,"B. Scientific data that will be preserved and shared, and the rationale for doing so:"
5,answer.text,answer.text,,"As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Repository sitting behavior metrics or analysis that were created in this project from the following UCSD-based study datasets: (i) The RISE study (ii) The iWATCH study (iii) NHANES cohorts Sufficient data from those datasets will be preserved to enable sharing to validate and replicate research findings described in the Aims with those datasets only. Please see Element 5 for data access information about other studies."
6,question.text,question.text,,"C. Metadata, other relevant data, and associated documentation:"
7,answer.text,answer.text,,"In addition to the data described above, code and models will be included in the repository and in on our project's GitHub website, hosted as part of the UCSD Advance Data Analytics Lab GitHub and titled ""Deep Postures""."
8,section.title,section.title,,"Element 2: Related Tools, Software and/or Code:"
9,answer.text,answer.text,,"Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using ActiLife software which requires a paid license. All custom code and models will also be shared in the data repository and on our project's GitHub website, hosted as part of the UCSD Advance Data Analytics Lab GitHub and titled ""Deep Postures""."


## llama3.3:70b


In [4]:
show('llama3.3:70b')


28 items in the annotation
  27 correct
   2 wrong
  f1 = 0.947

Every item, annotation against model:


,annotation says,model said,ok,text
0,title,title,,DATA MANAGEMENT AND SHARING PLAN
1,section.title,section.title,,Element 1: Data Type:
2,question.text,question.text,,A. Types and amount of scientific data expected to be generated in the project:
3,answer.text,answer.text,,"This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and the publicly available NHANES cohorts (wrist NHANES 2011-2014; hip/counts-NHANES 2003-2006). The studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the iWATCH Study, (iv) the MOCA Study, (v) the PHASE Study, (vi) the AusDiab Study, (vii) the ACT Study, and (viii) the WHISH accelerometer substudy."
4,question.text,question.text,,"B. Scientific data that will be preserved and shared, and the rationale for doing so:"
5,answer.text,answer.text,,"As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Repository sitting behavior metrics or analysis that were created in this project from the following UCSD-based study datasets: (i) The RISE study (ii) The iWATCH study (iii) NHANES cohorts Sufficient data from those datasets will be preserved to enable sharing to validate and replicate research findings described in the Aims with those datasets only. Please see Element 5 for data access information about other studies."
6,question.text,question.text,,"C. Metadata, other relevant data, and associated documentation:"
7,answer.text,answer.text,,"In addition to the data described above, code and models will be included in the repository and in on our project's GitHub website, hosted as part of the UCSD Advance Data Analytics Lab GitHub and titled ""Deep Postures""."
8,section.title,section.title,,"Element 2: Related Tools, Software and/or Code:"
9,answer.text,answer.text,,"Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using ActiLife software which requires a paid license. All custom code and models will also be shared in the data repository and on our project's GitHub website, hosted as part of the UCSD Advance Data Analytics Lab GitHub and titled ""Deep Postures""."


Plus 1 item(s) the model produced that the annotation does not contain:


,model said,text
0,answer.text,Objective sedentary behavior metrics – no existing standards


---

**How the two are lined up**

Each row pairs one annotation item with the label the model gave *that same text*.
The pairing is done by shared words, not by position — the model usually produces a
different number of items, so the two lists drift out of step and comparing row 12
against row 12 would compare unrelated things.

| column | meaning |
|---|---|
| `annotation says` | the correct label |
| `model said` | what the model called that text, or `not produced` if it never appeared |
| `ok` | `X` marks a row where the two disagree |

Items listed separately at the end are ones the model produced that have no
counterpart in the annotation at all.
